In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

Table Creation: Accounts

In [3]:
cursor.execute("""
CREATE TABLE accounts (
    account_id INTEGER PRIMARY KEY,
    account_name TEXT,
    industry TEXT,
    signup_date TEXT,
    region TEXT
)

""")

Table Creation: Users

In [4]:
cursor.execute("""
CREATE TABLE users (
    user_id INTEGER PRIMARY KEY,
    account_id INTEGER,
    name TEXT,
    role TEXT,
    signup_date TEXT,
    FOREIGN KEY(account_id) REFERENCES accounts(account_id)
);
""")

Table Creation: Plans

In [5]:
cursor.execute("""
CREATE TABLE plans (
    plan_id INTEGER PRIMARY KEY,
    plan_name TEXT,
    monthly_price REAL
);
""")

Table Creation: Subscriptions

In [6]:
cursor.execute("""
CREATE TABLE subscriptions (
    subscription_id INTEGER PRIMARY KEY,
    account_id INTEGER,
    plan_id INTEGER,
    start_date TEXT,
    end_date TEXT,
    status TEXT,
    FOREIGN KEY(account_id) REFERENCES accounts(account_id),
    FOREIGN KEY(plan_id) REFERENCES plans(plan_id)
);
""")

Table Creation: invoices

In [7]:
cursor.execute("""
CREATE TABLE invoices (
    invoice_id INTEGER PRIMARY KEY,
    account_id INTEGER,
    amount REAL,
    invoice_date TEXT,
    status TEXT,
    FOREIGN KEY(account_id) REFERENCES accounts(account_id)
);
""")

Table creation: payments

In [8]:
cursor.execute("""
CREATE TABLE payments (
    payment_id INTEGER PRIMARY KEY,
    invoice_id INTEGER,
    amount REAL,
    payment_date TEXT,
    status TEXT,
    FOREIGN KEY(invoice_id) REFERENCES invoices(invoice_id)
);
""")

Table Creation: features

In [9]:
cursor.execute("""
CREATE TABLE features (
    feature_id INTEGER PRIMARY KEY,
    feature_name TEXT
);
""")

Table Creation: feature_usage

In [10]:
cursor.execute("""
CREATE TABLE feature_usage (
    usage_id INTEGER PRIMARY KEY,
    user_id INTEGER,
    feature_id INTEGER,
    usage_date TEXT,
    usage_count INTEGER,
    FOREIGN KEY(user_id) REFERENCES users(user_id),
    FOREIGN KEY(feature_id) REFERENCES features(feature_id)
);
""")

Table Creation: deals

In [11]:
cursor.execute("""
CREATE TABLE deals (
    deal_id INTEGER PRIMARY KEY,
    account_id INTEGER,
    deal_value REAL,
    stage TEXT,
    close_date TEXT,
    FOREIGN KEY(account_id) REFERENCES accounts(account_id)
);
""")

Table Creation: sales_rep

In [12]:
cursor.execute("""
CREATE TABLE sales_reps (
    rep_id INTEGER PRIMARY KEY,
    name TEXT,
    region TEXT
);
""")

Create Table: supports_tickets

In [13]:
cursor.execute("""
CREATE TABLE support_tickets (
    ticket_id INTEGER PRIMARY KEY,
    account_id INTEGER,
    issue_type TEXT,
    status TEXT,
    created_date TEXT,
    resolved_date TEXT,
    FOREIGN KEY(account_id) REFERENCES accounts(account_id)
);
""")

Running Python File

In [14]:
!python saas_data_generator.py

🚀 Starting SaaS data generation...
✅ Plans:          3
✅ Sales Reps:     20
✅ Accounts:       750
✅ Users:          7500
✅ Subscriptions:  750
   Generating payments for 13477 invoices...
✅ Invoices:       13477
✅ Payments:       11456
✅ Features:       12
✅ Feature Usage:  57447
✅ Deals:          750
✅ Support Tickets:2046

🎉 Done! Database saved as: saas_database.db
   Ready for SQL practice and Power BI import.


In [15]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("saas_database.db")

In [16]:
pd.read_sql("SELECT COUNT(*) FROM accounts", conn)
pd.read_sql("SELECT COUNT(*) FROM users", conn)
pd.read_sql("SELECT COUNT(*) FROM invoices", conn)
pd.read_sql("SELECT COUNT(*) FROM payments", conn)

,COUNT(*)
0,11456


In [17]:
import sqlite3
conn = sqlite3.connect("saas_database.db")
cur = conn.cursor()

# Get all table names
cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cur.fetchall()

# Loop and print columns for each
for (table_name,) in tables:
    cur.execute(f"PRAGMA table_info({table_name})")
    columns = cur.fetchall()
    print(f"\n📋 {table_name}")
    for col in columns:
        print(f"   {col[1]} ({col[2]})")  # col[1]=name, col[2]=datatype


📋 plans
   plan_id (INTEGER)
   plan_name (TEXT)
   price (REAL)
   billing_cycle (TEXT)

📋 accounts
   account_id (INTEGER)
   company_name (TEXT)
   industry (TEXT)
   country (TEXT)
   signup_date (TEXT)
   status (TEXT)

📋 users
   user_id (INTEGER)
   account_id (INTEGER)
   full_name (TEXT)
   email (TEXT)
   role (TEXT)
   created_at (TEXT)
   is_active (INTEGER)

📋 subscriptions
   subscription_id (INTEGER)
   account_id (INTEGER)
   plan_id (INTEGER)
   start_date (TEXT)
   end_date (TEXT)
   status (TEXT)

📋 invoices
   invoice_id (INTEGER)
   account_id (INTEGER)
   amount (REAL)
   issue_date (TEXT)
   due_date (TEXT)
   status (TEXT)

📋 payments
   payment_id (INTEGER)
   invoice_id (INTEGER)
   amount_paid (REAL)
   payment_date (TEXT)
   method (TEXT)
   status (TEXT)

📋 features
   feature_id (INTEGER)
   feature_name (TEXT)
   category (TEXT)

📋 feature_usage
   usage_id (INTEGER)
   user_id (INTEGER)
   feature_id (INTEGER)
   usage_date (TEXT)
   usage_count (INTEGE

Problem 1 (Core Interview Question)

**“What is the total revenue collected per account?”**

In [18]:
pd.read_sql("""
SELECT
  i.account_id,
  SUM(p.amount_paid) AS total_revenue
FROM payments p
JOIN invoices i
  ON p.invoice_id = i.invoice_id
GROUP BY i.account_id
LIMIT 15
""", conn)

,account_id,total_revenue
0,1,1358.77
1,2,343.00
2,3,2682.00
3,4,9631.93
4,5,637.00
5,6,2508.12
6,7,294.00
7,8,392.00
8,9,245.00
9,10,8881.38


**“Show ALL accounts and their revenue, including those with 0 revenue”**

In [19]:
pd.read_sql("""
SELECT
  i.account_id,
  COUNT(*) AS total_payments,
  SUM(p.amount_paid) AS total_revenue
FROM payments p
LEFT JOIN invoices i
  ON p.invoice_id = i.invoice_id
GROUP BY i.account_id
LIMIT 15
""", conn)

,account_id,total_payments,total_revenue
0,1,30,1358.77
1,2,7,343.00
2,3,18,2682.00
3,4,20,9631.93
4,5,13,637.00
5,6,18,2508.12
6,7,6,294.00
7,8,8,392.00
8,9,5,245.00
9,10,18,8881.38


**“Show top 5 accounts by revenue”**

In [20]:
pd.read_sql(
    """
    SELECT
      i.account_id,
      COUNT(*) AS total_payments,
      SUM(p.amount_paid) AS total_revenue
    FROM payments p
    JOIN invoices i
      ON p.invoice_id = i.invoice_id
    GROUP BY i.account_id
    ORDER BY total_revenue DESC
    LIMIT 10
    """,conn
)

,account_id,total_payments,total_revenue
0,67,31,15315.17
1,562,31,14998.97
2,722,32,14962.79
3,354,32,14524.56
4,377,30,14228.30
5,638,29,14028.38
6,196,29,13646.93
7,703,28,13215.24
8,664,27,13200.34
9,149,27,13197.79


**“Show top 5 accounts by revenue WITH account name”**

In [21]:
pd.read_sql("""
SELECT
  a.account_id,
  a.company_name,
  COUNT(p.payment_id) AS total_payments,
  COALESCE(SUM(p.amount_paid), 0) AS total_revenue
FROM accounts a
LEFT JOIN invoices i
  ON a.account_id = i.account_id
LEFT JOIN payments p
  ON i.invoice_id = p.invoice_id
GROUP BY a.account_id, a.company_name
ORDER BY total_revenue DESC
LIMIT 10
""", conn)

,account_id,company_name,total_payments,total_revenue
0,67,Brown Corp,31,15315.17
1,562,Wilson Solutions,31,14998.97
2,722,Johnson Innovations,32,14962.79
3,354,Kumar Innovations,32,14524.56
4,377,Davis Innovations,30,14228.30
5,638,Garcia Labs,29,14028.38
6,196,Sharma Labs,29,13646.93
7,703,Patel Innovations,28,13215.24
8,664,Nair Systems,27,13200.34
9,149,Sharma Solutions,27,13197.79


In [22]:
pd.read_sql(
    """
    SELECT
      a.account_id,
      a.company_name,
      COUNT(p.payment_id) as total_payments,
      COALESCE(SUM(p.amount_paid), 0) AS total_revenue
    FROM accounts a
    LEFT JOIN invoices i
      on a.account_id = i.account_id
    LEFT JOIN payments p
      ON i.invoice_id = p.invoice_id
    GROUP BY a.account_id, a.company_name
    ORDER BY total_revenue DESC
    LIMIT 10
    """, conn
)

,account_id,company_name,total_payments,total_revenue
0,67,Brown Corp,31,15315.17
1,562,Wilson Solutions,31,14998.97
2,722,Johnson Innovations,32,14962.79
3,354,Kumar Innovations,32,14524.56
4,377,Davis Innovations,30,14228.30
5,638,Garcia Labs,29,14028.38
6,196,Sharma Labs,29,13646.93
7,703,Patel Innovations,28,13215.24
8,664,Nair Systems,27,13200.34
9,149,Sharma Solutions,27,13197.79


**“Compare revenue from ACTIVE vs CHURNED accounts”**

In [23]:
pd.read_sql(
    """
    SELECT
      a.status,
      SUM(p.amount_paid) as total_revenue
    FROM accounts a
    JOIN invoices i
      ON a.account_id = i.account_id
    JOIN payments p
      ON i.invoice_id = p.invoice_id
    GROUP BY a.status

    """,conn
)

,status,total_revenue
0,active,1557384.97
1,churned,124343.60


**“What is the average revenue per account for active vs churned?”**

In [24]:
pd.read_sql(
    """
    SELECT
      a.status,
      COALESCE(AVG(p.amount_paid), 0) as avg_revenue
    FROM accounts a
    LEFT JOIN invoices i
      ON a.account_id = i.account_id
    LEFT JOIN payments p
      ON i.invoice_id = p.invoice_id
    GROUP BY a.status

    """,conn
)

,status,avg_revenue
0,active,146.978574
1,churned,144.585581


**Same Solution with Inner Query**

In [25]:
pd.read_sql("""
SELECT
    status,
    AVG(total_revenue) AS avg_revenue_per_account
FROM (
    SELECT
        a.account_id,
        a.status,
        COALESCE(SUM(p.amount_paid), 0) AS total_revenue
    FROM accounts a
    LEFT JOIN invoices i
        ON a.account_id = i.account_id
    LEFT JOIN payments p
        ON i.invoice_id = p.invoice_id
    GROUP BY a.account_id, a.status
) t
GROUP BY status
""", conn)

,status,avg_revenue_per_account
0,active,2675.919192
1,churned,740.140476


**Which features are most used by high-revenue accounts?**

Step 1:

Find high revenue accounts

👉 Use:
revenue per account (you already did this)

Step 2:

Get users of those accounts
👉 accounts → users

Step 3:

Get feature usage
👉 users → feature_usage → features

In [26]:
pd.read_sql(
    """
    SELECT
      f.feature_name,
      SUM(fu.usage_count) AS total_usage
    FROM users u
    JOIN feature_usage fu
      ON u.user_id = fu.user_id
    JOIN features f
      ON fu.feature_id = f.feature_id
    WHERE u.account_id IN(
        SELECT
          a.account_id
        FROM accounts a
        LEFT JOIN invoices i
          ON a.account_id = i.account_id
        LEFT JOIN payments p
          ON i.invoice_id = p.invoice_id
        GROUP BY a.account_id
        HAVING SUM(p.amount_paid) > 3000
    )
    GROUP BY f.feature_name
    ORDER BY total_usage DESC
    LIMIT 5

    """, conn
)

,feature_name,total_usage
0,CSV Export,57295
1,AI Insights,56543
2,Team Collaboration,56311
3,Custom Reports,56272
4,Dashboard Analytics,55653


**Practice Exercise**

Problem 1 (Warm-up)

**“Show accounts that have made more than 3 payments”**

In [27]:
pd.read_sql(
    """
    SELECT
      a.account_id,
      a.company_name,
      COUNT(p.payment_id) AS total_time_paid
    FROM accounts a
    LEFT JOIN invoices i
      ON a.account_id = i.account_id
    LEFT JOIN payments p
      ON i.invoice_id = p.invoice_id
    GROUP BY a.account_id, a.company_name
    HAVING COUNT(p.payment_id) > 3

    LIMIT 10

    """,conn
)

,account_id,company_name,total_time_paid
0,1,Wilson Group,30
1,2,Kumar Corp,7
2,3,Gupta Services,18
3,4,Johnson Systems,20
4,5,Verma Solutions,13
5,6,Kumar Systems,18
6,7,Davis Group,6
7,8,Patel Group,8
8,9,Verma Solutions,5
9,10,Smith Group,18


Problem 2 (Slightly harder)

**“Show accounts where total revenue is more than 5000”**

In [28]:
pd.read_sql(
    """
    SELECT
      a.account_id,
      a.company_name,
      SUM(p.amount_paid) as total_revenue
    FROM accounts a
    LEFT JOIN invoices i
      ON a.account_id = i.account_id
    LEFT JOIN payments p
      ON i.invoice_id = p.invoice_id
    GROUP BY a.account_id, a.company_name
    HAVING SUM(p.amount_paid) > 5000
    LIMIT 10

    """,conn
)

,account_id,company_name,total_revenue
0,4,Johnson Systems,9631.93
1,10,Smith Group,8881.38
2,60,Johnson Solutions,10019.01
3,67,Brown Corp,15315.17
4,83,Sharma Solutions,12635.08
5,85,Smith Services,8982.00
6,90,Nair Labs,6986.00
7,92,Williams Systems,7290.16
8,112,Mehta Group,9664.22
9,118,Brown Innovations,10905.09


Problem 3 (Important twist)

**“Show payments greater than 500”**

In [29]:
pd.read_sql(
    """
  SELECT *
  FROM payments
  WHERE amount_paid > 500
  ORDER BY amount_paid DESC;
    """,conn
)

,payment_id,invoice_id,amount_paid,payment_date,method,status


**“Show accounts that have generated more than 5 invoices”**

In [30]:
pd.read_sql(
    """
    SELECT
      a.account_id,
      a.company_name,
      COUNT(i.invoice_id) as total_invoices
    FROM accounts a
    JOIN invoices i
      ON a.account_id = i.account_id
    GROUP BY a.account_id, a.company_name
    HAVING COUNT(i.invoice_id) > 5

    """,conn
)

,account_id,company_name,total_invoices
0,1,Wilson Group,34
1,2,Kumar Corp,8
2,3,Gupta Services,21
3,4,Johnson Systems,25
4,5,Verma Solutions,15
...,...,...,...
658,746,Patel Digital,16
659,747,Nair Innovations,7
660,748,Patel Solutions,24
661,749,Joshi Labs,15


**“Show accounts that have total revenue between 2000 and 5000”**

In [31]:
pd.read_sql(
    """
    SELECT
      a.account_id,
      a.company_name,
      SUM(p.amount_paid) as total_revenue
    FROM accounts a
    JOIN invoices i
      ON a.account_id = i.account_id
    JOIN payments p
      ON i.invoice_id = p.invoice_id
    GROUP BY a.account_id, a.company_name
    HAVING SUM(p.amount_paid) BETWEEN 2000 AND 5000

    """,conn
)

,account_id,company_name,total_revenue
0,3,Gupta Services,2682.00
1,6,Kumar Systems,2508.12
2,16,Brown Digital,2913.78
3,35,Singh Labs,3091.99
4,37,Wilson Digital,3784.62
...,...,...,...
142,714,Nair Digital,2428.96
143,718,Sharma Enterprises,3992.00
144,726,Williams Enterprises,2302.97
145,742,Smith Solutions,2737.02


**“Show accounts that have made payments in more than 2 different months”**

In [32]:
pd.read_sql(
    """
    SELECT
      a.account_id,
      a.company_name
    FROM accounts a
    JOIN invoices i
      ON a.account_id = i.account_id
    JOIN payments p
      ON i.invoice_id = p.invoice_id
    GROUP BY a.account_id, a.company_name
    HAVING COUNT(DISTINCT strftime('%Y-%m', p.payment_date)) > 2
    LIMIT 15
    """,conn
)

,account_id,company_name
0,1,Wilson Group
1,2,Kumar Corp
2,3,Gupta Services
3,4,Johnson Systems
4,5,Verma Solutions
5,6,Kumar Systems
6,7,Davis Group
7,8,Patel Group
8,9,Verma Solutions
9,10,Smith Group


**“Show users who belong to high-revenue accounts (>5000)”**

In [33]:
pd.read_sql(
    """
  SELECT
    user_id,
    full_name,
    is_active
  FROM users u
  WHERE u.account_id IN (
    SELECT
      a.account_id
    FROM accounts a
    JOIN invoices i
      ON a.account_id = i.account_id
    JOIN payments p
      ON i.invoice_id = p.invoice_id
    GROUP BY a.account_id
    HAVING SUM(p.amount_paid) > 5000
);

    """,conn
)

,user_id,full_name,is_active
0,35,Daniel Singh,1
1,36,Michael Smith,1
2,37,Rohan Kumar,1
3,38,Arjun Gupta,1
4,39,Siddharth Jones,1
...,...,...,...
800,7289,Laura Gupta,1
801,7290,Harsh Jones,1
802,7291,Sarah Patel,1
803,7292,James Verma,1


**“Show average revenue per account”**

In [34]:
pd.read_sql("""
SELECT
    status,
    AVG(total_revenue) AS avg_revenue_per_account
FROM (
    SELECT
        a.account_id,
        a.status,
        COALESCE(SUM(p.amount_paid), 0) AS total_revenue
    FROM accounts a
    LEFT JOIN invoices i
        ON a.account_id = i.account_id
    LEFT JOIN payments p
        ON i.invoice_id = p.invoice_id
    GROUP BY a.account_id, a.status
) t
GROUP BY status
""", conn)


,status,avg_revenue_per_account
0,active,2675.919192
1,churned,740.140476


**“Show accounts that have at least one payment”**

In [35]:
pd.read_sql("""
SELECT
    a.account_id,
    a.company_name
FROM accounts a
WHERE EXISTS (
    SELECT 1
    FROM invoices i
    JOIN payments p
        ON i.invoice_id = p.invoice_id
    WHERE i.account_id = a.account_id
)
""", conn)

,account_id,company_name
0,1,Wilson Group
1,2,Kumar Corp
2,3,Gupta Services
3,4,Johnson Systems
4,5,Verma Solutions
...,...,...
741,746,Patel Digital
742,747,Nair Innovations
743,748,Patel Solutions
744,749,Joshi Labs


Problem

“Show top 5 accounts (with name) that:

have made payments in more than 2 different months

have total revenue greater than 5000
and show:

  total revenue

  number of payments
  
  number of active users in that account”

In [36]:
pd.read_sql("""
SELECT
    a.account_id,
    a.company_name,
    p.total_revenue,
    p.total_payments,
    u.active_users
FROM (
    -- 🔹 Payment metrics
    SELECT
        i.account_id,
        SUM(p.amount_paid) AS total_revenue,
        COUNT(p.payment_id) AS total_payments,
        COUNT(DISTINCT strftime('%Y-%m', p.payment_date)) AS active_months
    FROM invoices i
    JOIN payments p
        ON i.invoice_id = p.invoice_id
    GROUP BY i.account_id
    HAVING
        SUM(p.amount_paid) > 5000
        AND COUNT(DISTINCT strftime('%Y-%m', p.payment_date)) > 2
) p
JOIN (
    -- 🔹 User metrics
    SELECT
        account_id,
        COUNT(user_id) AS active_users
    FROM users
    WHERE is_active = 1
    GROUP BY account_id
) u
    ON p.account_id = u.account_id
JOIN accounts a
    ON a.account_id = p.account_id
ORDER BY p.total_revenue DESC
LIMIT 5
""", conn)

,account_id,company_name,total_revenue,total_payments,active_users
0,67,Brown Corp,15315.17,31,20
1,562,Wilson Solutions,14998.97,31,18
2,354,Kumar Innovations,14524.56,32,20
3,377,Davis Innovations,14228.30,30,5
4,196,Sharma Labs,13646.93,29,16


**Window Functions (Simple Meaning)**

Window functions perform calculations across rows without grouping them

**First Window Function: ROW_NUMBER()**

“Give row number to each payment per account (latest first)”

In [37]:
pd.read_sql(
    """
    SELECT
      i.account_id,
      p.payment_id,
      p.amount_paid,
      p.payment_date,
      ROW_NUMBER() OVER(
        PARTITION BY i.account_id
        ORDER BY p.payment_date DESC
      ) AS rn
    FROM payments p
    JOIN invoices i
      ON p.invoice_id = i.invoice_id

    """,conn
)

,account_id,payment_id,amount_paid,payment_date,rn
0,1,30,49.00,2024-12-14,1
1,1,29,49.00,2024-10-22,2
2,1,28,49.00,2024-09-27,3
3,1,27,49.00,2024-08-14,4
4,1,26,49.00,2024-07-18,5
...,...,...,...,...,...
11451,750,11453,499.00,2024-09-24,4
11452,750,11452,229.06,2024-08-27,5
11453,750,11451,499.00,2024-08-06,6
11454,750,11450,499.00,2024-06-24,7


You can now solve:

**“Get latest payment per account”**

In [38]:
pd.read_sql("""
SELECT *
FROM (
    SELECT
        i.account_id,
        p.payment_id,
        p.amount_paid,
        p.payment_date,
        ROW_NUMBER() OVER (
            PARTITION BY i.account_id
            ORDER BY p.payment_date DESC
        ) AS rn
    FROM payments p
    JOIN invoices i
        ON p.invoice_id = i.invoice_id
) t
WHERE rn = 1
LIMIT 10
""", conn)

,account_id,payment_id,amount_paid,payment_date,rn
0,1,30,49.0,2024-12-14,1
1,2,37,49.0,2024-12-31,1
2,3,55,149.0,2024-12-06,1
3,4,75,499.0,2024-12-31,1
4,5,88,49.0,2024-12-15,1
5,6,106,149.0,2024-12-31,1
6,7,112,49.0,2023-09-12,1
7,8,120,49.0,2024-12-06,1
8,9,125,49.0,2023-12-26,1
9,10,143,499.0,2024-12-30,1


**ROW_NUMBER vs RANK vs DENSE_RANK**

“Rank payments per account based on amount”

In [55]:
pd.read_sql("""
SELECT
    i.account_id,
    p.payment_id,
    p.amount_paid,

    ROW_NUMBER() OVER (
        PARTITION BY i.account_id
        ORDER BY p.payment_id DESC
    ) AS row_num,

    RANK() OVER (
        PARTITION BY i.account_id
        ORDER BY p.payment_id DESC
    ) AS rank_num,

    DENSE_RANK() OVER (
        PARTITION BY i.account_id
        ORDER BY p.payment_id DESC
    ) AS dense_rank_num

FROM payments p
JOIN invoices i
    ON p.invoice_id = i.invoice_id
LIMIT 20
""", conn)

,account_id,payment_id,amount_paid,row_num,rank_num,dense_rank_num
0,1,30,49.00,1,1,1
1,1,29,49.00,2,2,2
2,1,28,49.00,3,3,3
3,1,27,49.00,4,4,4
4,1,26,49.00,5,5,5
5,1,25,49.00,6,6,6
6,1,24,49.00,7,7,7
7,1,23,49.00,8,8,8
8,1,22,49.00,9,9,9
9,1,21,35.04,10,10,10


**“Get top 2 highest payments per account”**

In [40]:
pd.read_sql("""
SELECT *
FROM (
    SELECT
        i.account_id,
        p.payment_id,
        p.amount_paid,
        p.amount_paid,
        ROW_NUMBER() OVER (
            PARTITION BY i.account_id
            ORDER BY p.amount_paid DESC
        ) AS rn
    FROM payments p
    JOIN invoices i
        ON p.invoice_id = i.invoice_id
)
WHERE rn  <= 2
LIMIT 10
""", conn)

,account_id,payment_id,amount_paid,amount_paid:1,rn
0,1,1,49.0,49.0,1
1,1,2,49.0,49.0,2
2,2,31,49.0,49.0,1
3,2,32,49.0,49.0,2
4,3,38,149.0,149.0,1
5,3,39,149.0,149.0,2
6,4,56,499.0,499.0,1
7,4,57,499.0,499.0,2
8,5,76,49.0,49.0,1
9,5,77,49.0,49.0,2


“For each account, show the top 2 payments (highest amount) ONLY for accounts where:

total revenue > 5000
and they have payments in more than 2 months

Also include:

account name

total revenue

payment rank”

In [41]:
pd.read_sql(
    """
    WITH account_metrics AS(
      SELECT
        i.account_id,
        SUM(p.amount_paid) AS total_revenue,
        COUNT(DISTINCT strftime('%Y-%m', p.payment_date)) AS active_months
      FROM invoices i
      JOIN payments p
        ON i.invoice_id = p.invoice_id
      GROUP BY i.account_id
      HAVING
        SUM(p.amount_paid) > 5000
        AND COUNT(DISTINCT strftime('%Y-%m', p.payment_date)) > 2
    ),

    ranked_payments AS(
      SELECT
        i.account_id,
        p.payment_id,
        p.amount_paid,
        p.payment_date,
        ROW_NUMBER() OVER (
          PARTITION BY i.account_id
          ORDER BY p.amount_paid DESC
        ) AS rn
    FROM payments p
    JOIN invoices i
      ON p.invoice_id = i.invoice_id
    )
    SELECT
      a.account_id,
      a.company_name,
      rp.payment_id,
      rp.amount_paid,
      am.total_revenue,
      rp.rn
    FROM ranked_payments rp
    JOIN account_metrics am
      ON rp.account_id = am.account_id
    JOIN accounts a
      ON a.account_id = rp.account_id
    WHERE rp.rn <= 2
    ORDER BY am.total_revenue DESC, rp.rn
    LIMIT 10
""", conn
)

,account_id,company_name,payment_id,amount_paid,total_revenue,rn
0,67,Brown Corp,1032,499.0,15315.17,1
1,67,Brown Corp,1033,499.0,15315.17,2
2,562,Wilson Solutions,8741,499.0,14998.97,1
3,562,Wilson Solutions,8742,499.0,14998.97,2
4,722,Johnson Innovations,11105,499.0,14962.79,1
5,722,Johnson Innovations,11106,499.0,14962.79,2
6,354,Kumar Innovations,5383,499.0,14524.56,1
7,354,Kumar Innovations,5384,499.0,14524.56,2
8,377,Davis Innovations,5802,499.0,14228.30,1
9,377,Davis Innovations,5804,499.0,14228.30,2


In [42]:

pd.read_sql(
    """
-- 🎯 Goal:
-- For each account:
-- - Only include accounts with:
--     • total revenue > 5000
--     • payments in more than 2 months
-- - Show top 2 highest payments per account
-- - Include account name and total revenue

WITH account_metrics AS (

-- 🔹 Step 1: Calculate account-level metrics
-- This avoids duplication and gives us clean aggregation

SELECT
    i.account_id,

    -- Total revenue collected from payments
    SUM(p.amount_paid) AS total_revenue,

    -- Number of distinct months with payments
    COUNT(DISTINCT strftime('%Y-%m', p.payment_date)) AS active_months

FROM invoices i
JOIN payments p
    ON i.invoice_id = p.invoice_id

-- Group by account to aggregate correctly
GROUP BY i.account_id

-- 🔹 Filter only "valid" accounts
HAVING
    SUM(p.amount_paid) > 5000
    AND COUNT(DISTINCT strftime('%Y-%m', p.payment_date)) > 2
),

ranked_payments AS (
-- 🔹 Step 2: Rank payments within each account

SELECT
    i.account_id,
    p.payment_id,
    p.amount_paid,
    p.payment_date,

    -- Assign rank based on payment amount (highest first)
    ROW_NUMBER() OVER (
        PARTITION BY i.account_id   -- Reset ranking per account
        ORDER BY p.amount_paid DESC -- Highest payment gets rank 1
    ) AS rn

FROM payments p
JOIN invoices i
    ON p.invoice_id = i.invoice_id
)

-- 🔹 Step 3: Final result combining everything

SELECT
  a.account_id,
  a.company_name,
-- Payment details
  rp.payment_id,
  rp.amount_paid,

-- Account-level metric
  am.total_revenue,

-- Rank of payment within account
  rp.rn

FROM ranked_payments rp

-- Join only valid accounts (filtered in CTE)
JOIN account_metrics am
  ON rp.account_id = am.account_id

-- Join account info (for name)
JOIN accounts a
  ON a.account_id = rp.account_id

-- 🔹 Keep only top 2 payments per account
WHERE rp.rn <= 2

-- 🔹 Order results:
-- First by highest revenue accounts
-- Then by payment rank within each account
ORDER BY
  am.total_revenue DESC,
  rp.rn

-- Limit output for readability
LIMIT 10

""",conn
)


,account_id,company_name,payment_id,amount_paid,total_revenue,rn
0,67,Brown Corp,1032,499.0,15315.17,1
1,67,Brown Corp,1033,499.0,15315.17,2
2,562,Wilson Solutions,8741,499.0,14998.97,1
3,562,Wilson Solutions,8742,499.0,14998.97,2
4,722,Johnson Innovations,11105,499.0,14962.79,1
5,722,Johnson Innovations,11106,499.0,14962.79,2
6,354,Kumar Innovations,5383,499.0,14524.56,1
7,354,Kumar Innovations,5384,499.0,14524.56,2
8,377,Davis Innovations,5802,499.0,14228.30,1
9,377,Davis Innovations,5804,499.0,14228.30,2


**LAG and LEAD**

Simple Definition

LAG → look at previous row
LEAD → look at next row

In [43]:
pd.read_sql("""
SELECT
    i.account_id,
    p.payment_date,
    p.amount_paid,

    -- Previous payment amount
    LAG(p.amount_paid) OVER (
        PARTITION BY i.account_id
        ORDER BY p.payment_date
    ) AS prev_payment

FROM payments p
JOIN invoices i
    ON p.invoice_id = i.invoice_id

LIMIT 20
""", conn)

,account_id,payment_date,amount_paid,prev_payment
0,1,2022-03-28,49.00,NaN
1,1,2022-05-05,49.00,49.00
2,1,2022-05-23,49.00,49.00
3,1,2022-07-07,32.60,49.00
4,1,2022-08-07,49.00,32.60
5,1,2022-09-07,49.00,49.00
6,1,2022-09-18,49.00,49.00
7,1,2022-11-07,49.00,49.00
8,1,2022-12-07,36.31,49.00
9,1,2023-01-26,49.00,36.31


**“Show payment change compared to previous payment”**

In [44]:
pd.read_sql(
    """
    SELECT
      i.account_id,
      p.payment_date,
      p.amount_paid,

      LAG(p.amount_paid) OVER(
        PARTITION BY i.account_id
        ORDER BY p.payment_date
      ) AS pre_payment,

      --DIFFERENCE
      p.amount_paid - LAG(p.amount_paid) OVER (
        PARTITION BY i.account_id
        ORDER BY p.payment_date
    ) AS change_amount
    FROM payments p
    JOIN invoices i
      ON p.invoice_id = i.invoice_id

    LIMIT 20
    """,conn
)

,account_id,payment_date,amount_paid,pre_payment,change_amount
0,1,2022-03-28,49.00,NaN,NaN
1,1,2022-05-05,49.00,49.00,0.00
2,1,2022-05-23,49.00,49.00,0.00
3,1,2022-07-07,32.60,49.00,-16.40
4,1,2022-08-07,49.00,32.60,16.40
5,1,2022-09-07,49.00,49.00,0.00
6,1,2022-09-18,49.00,49.00,0.00
7,1,2022-11-07,49.00,49.00,0.00
8,1,2022-12-07,36.31,49.00,-12.69
9,1,2023-01-26,49.00,36.31,12.69


**LEAD**
“Show next payment amount”

In [45]:
pd.read_sql("""
SELECT
    i.account_id,
    p.payment_date,
    p.amount_paid,

    LEAD(p.amount_paid) OVER (
        PARTITION BY i.account_id
        ORDER BY p.payment_date
    ) AS next_payment

FROM payments p
JOIN invoices i
    ON p.invoice_id = i.invoice_id

LIMIT 20
""", conn)

,account_id,payment_date,amount_paid,next_payment
0,1,2022-03-28,49.00,49.00
1,1,2022-05-05,49.00,49.00
2,1,2022-05-23,49.00,32.60
3,1,2022-07-07,32.60,49.00
4,1,2022-08-07,49.00,49.00
5,1,2022-09-07,49.00,49.00
6,1,2022-09-18,49.00,49.00
7,1,2022-11-07,49.00,36.31
8,1,2022-12-07,36.31,49.00
9,1,2023-01-26,49.00,49.00




**👉 “Show only payments where amount increased compared to previous payment”**

In [46]:
pd.read_sql(
    """
    SELECT *
    FROM(
      SELECT
        i.account_id,
        p.payment_date,
        p.amount_paid,

        LAG(p.amount_paid) OVER(
          PARTITION BY i.account_id
          ORDER BY p.payment_date
        )AS prev_payment
      FROM payments p
      JOIN invoices i
        ON p.invoice_id = i.invoice_id

    )
    WHERE amount_paid > prev_payment
    LIMIT 20

    """,conn
)

,account_id,payment_date,amount_paid,prev_payment
0,1,2022-08-07,49.0,32.60
1,1,2023-01-26,49.0,36.31
2,1,2023-04-16,49.0,38.44
3,1,2023-09-15,49.0,22.36
4,1,2023-12-18,49.0,18.02
5,1,2024-02-09,49.0,35.04
6,4,2023-05-23,499.0,349.10
7,4,2024-12-15,499.0,300.83
8,6,2023-08-04,149.0,56.44
9,6,2024-06-23,149.0,111.05


**CASE WHEN**

Simple Meaning

CASE WHEN = IF-ELSE logic in SQL

In [47]:
pd.read_sql(
    """
    SELECT
      amount_paid,
      CASE
        WHEN amount_paid > 500 THEN 'HIGH'
        WHEN amount_paid > 200 THEN 'Medium'
      END AS payment_category
    FROM payments
    LIMIT 100
    """,conn
)

,amount_paid,payment_category
0,49.0,None
1,49.0,None
2,49.0,None
3,32.6,None
4,49.0,None
...,...,...
95,149.0,None
96,149.0,None
97,149.0,None
98,149.0,None


Problem 1

**“Classify accounts based on total revenue”**

In [48]:
pd.read_sql("""
SELECT
  a.account_id,
  a.company_name,
  SUM(p.amount_paid) AS total_revenue,

  CASE
    WHEN SUM(p.amount_paid) > 5000 THEN 'High'
    WHEN SUM(p.amount_paid) BETWEEN 2000 AND 5000 THEN 'Medium'
    ELSE 'Low'
  END AS revenue_category

FROM accounts a
LEFT JOIN invoices i
  ON a.account_id = i.account_id
LEFT JOIN payments p
  ON i.invoice_id = p.invoice_id

GROUP BY a.account_id, a.company_name
LIMIT 10
""", conn)

,account_id,company_name,total_revenue,revenue_category
0,1,Wilson Group,1358.77,Low
1,2,Kumar Corp,343.00,Low
2,3,Gupta Services,2682.00,Medium
3,4,Johnson Systems,9631.93,High
4,5,Verma Solutions,637.00,Low
5,6,Kumar Systems,2508.12,Medium
6,7,Davis Group,294.00,Low
7,8,Patel Group,392.00,Low
8,9,Verma Solutions,245.00,Low
9,10,Smith Group,8881.38,High


**“Count how many accounts fall into each category”**

In [49]:
pd.read_sql("""
SELECT
  revenue_category,
  COUNT(*) AS num_accounts
FROM(
  SELECT
    a.account_id,
    a.company_name,
    SUM(p.amount_paid) AS total_revenue,

    CASE
      WHEN SUM(p.amount_paid) > 5000 THEN 'High'
      WHEN SUM(p.amount_paid) BETWEEN 2000 AND 5000 THEN 'Medium'
      ELSE 'Low'
    END AS revenue_category

  FROM accounts a
  LEFT JOIN invoices i
    ON a.account_id = i.account_id
  LEFT JOIN payments p
    ON i.invoice_id = p.invoice_id

  GROUP BY a.account_id, a.company_name
) t
GROUP BY revenue_category

""", conn)

,revenue_category,num_accounts
0,High,78
1,Low,525
2,Medium,147


In [50]:
pd.read_sql("""
SELECT
    payment_category,
    COUNT(*) AS num_payments
FROM (
    SELECT
        payment_id,
        amount_paid,

        CASE
            WHEN amount_paid > 500 THEN 'High'
            WHEN amount_paid BETWEEN 200 AND 500 THEN 'Medium'
            ELSE 'Low'
        END AS payment_category

    FROM payments
) t
GROUP BY payment_category
""", conn)

,payment_category,num_payments
0,Low,9552
1,Medium,1904


**“Show total revenue per month”**

In [51]:
pd.read_sql("""
SELECT
  strftime('%Y-%m', p.payment_date) AS month,
  SUM(p.amount_paid) AS total_revenue
FROM payments p
GROUP BY month
ORDER BY month
""", conn)

,month,total_revenue
0,2022-01,842.00
1,2022-02,3055.89
2,2022-03,6448.78
3,2022-04,8599.82
4,2022-05,10685.86
5,2022-06,14573.75
6,2022-07,19033.07
7,2022-08,19146.50
8,2022-09,21423.05
9,2022-10,23954.93


**How many accounts made payments each month?**

In [52]:
pd.read_sql("""
SELECT
    strftime('%Y-%m', p.payment_date) AS month,
    COUNT(DISTINCT i.account_id) AS active_accounts
FROM payments p
JOIN invoices i
    ON p.invoice_id = i.invoice_id
GROUP BY month
ORDER BY month
""", conn)

,month,active_accounts
0,2022-01,8
1,2022-02,20
2,2022-03,45
3,2022-04,63
4,2022-05,78
5,2022-06,97
6,2022-07,123
7,2022-08,128
8,2022-09,142
9,2022-10,155


**“When did each account first pay?”**

In [53]:
pd.read_sql("""
SELECT
    i.account_id,
    MIN(p.payment_date) AS first_payment_date
FROM payments p
JOIN invoices i
    ON p.invoice_id = i.invoice_id
GROUP BY i.account_id
LIMIT 10
""", conn)

,account_id,first_payment_date
0,1,2022-03-28
1,2,2024-06-22
2,3,2023-05-05
3,4,2023-01-11
4,5,2023-11-03
5,6,2023-02-10
6,7,2022-12-20
7,8,2024-04-23
8,9,2023-08-16
9,10,2023-06-14


**Show monthly revenue growth (compare with previous month)**